# 🔎 Georgia RAG — пайплайн по шагам

Этот ноутбук позволяет руками пройти весь путь и посмотреть, что получается на каждом этапе:

1. Проверка `.env` и настроек
2. (опц.) Выгрузка истории чата
3. Сырые сообщения
4. Чанкинг (можно крутить параметры)
5. Индексация
6. Поиск (retrieve)
7. Ответ RAG
8. Дебаг: какой промпт уходит в GPT

> Запуск: `uv run jupyter lab` из корня проекта, затем открыть `notebooks/explore.ipynb`.

In [ ]:
# Чтобы import config / src.* работали из папки notebooks/ — переходим в корень проекта
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Рабочая папка:", os.getcwd())

# Автоперезагрузка: правки в src/*.py подхватываются без перезапуска ядра
%load_ext autoreload
%autoreload 2

## 1. Проверка `.env` и настроек

Убедимся, что ключи подхватились (значения замаскированы).

In [ ]:
import importlib, config
importlib.reload(config)

def mask(v):
    return (v[:4] + "…" + str(len(v)) + " симв.") if v else "❌ не задан"

print("OPENAI_API_KEY  :", mask(config.OPENAI_API_KEY))
print("BOT_TOKEN       :", mask(config.BOT_TOKEN))
print("TELEGRAM_API_ID :", config.TELEGRAM_API_ID or "❌ не задан")
print("TELEGRAM_PHONE  :", config.TELEGRAM_PHONE or "❌ не задан")
print()
print("Чаты:", [c["username"] for c in config.CHATS])
print("Модели:", config.EMBED_MODEL, "|", config.CHAT_MODEL)
print("Чанкинг: gap =", config.CHUNK_TIME_GAP_MIN, "мин, max =", config.CHUNK_MAX_CHARS, "симв.")

## 2. (опционально) Выгрузка истории чата

Если ты уже запускала `uv run python -m src.ingest` в терминале — пропусти этот шаг.

Первый запуск попросит код подтверждения из Telegram (вводится прямо в ноутбуке). После авторизации создаётся файл сессии, и повторно код не понадобится.

In [ ]:
# Раскомментируй, чтобы выгрузить историю прямо из ноутбука:
#
# from telethon import TelegramClient
# from src.ingest import ingest_chat
#
# client = TelegramClient("georgia_ingest", int(config.TELEGRAM_API_ID), config.TELEGRAM_API_HASH)
# await client.start(phone=config.TELEGRAM_PHONE or None)
# for chat in config.CHATS:
#     await ingest_chat(client, chat)
# await client.disconnect()

## 3. Сырые сообщения

Смотрим, что выгрузилось: сколько сообщений и как они выглядят.

In [ ]:
from src.preprocess import _load_raw

username = config.CHATS[0]["username"]
raw_path = config.RAW_DIR / f"{username}.jsonl"
print("Файл:", raw_path, "| есть:", raw_path.exists())

if raw_path.exists():
    msgs = _load_raw(raw_path)
    print("Всего сообщений:", len(msgs))
    print("\nПоследние 5:")
    for m in msgs[-5:]:
        sender = m.get("sender") or "Аноним"
        print(f"  [{m['date'][:16]}] {sender}: {m['text'][:90]}")
else:
    print("Сначала выгрузи историю (шаг 2 или `uv run python -m src.ingest`).")

## 4. Чанкинг — как сообщения склеиваются в диалоги

Собираем чанки и смотрим на результат. Параметры `CHUNK_TIME_GAP_MIN` и `CHUNK_MAX_CHARS` можно менять прямо здесь и сравнивать.

In [ ]:
from src.preprocess import chunk_messages

# Можно поэкспериментировать с параметрами:
# config.CHUNK_TIME_GAP_MIN = 15
# config.CHUNK_MAX_CHARS = 2000

chunks = chunk_messages(msgs)
sizes = [len(c["text"]) for c in chunks]
print(f"{len(msgs)} сообщений -> {len(chunks)} чанков")
if sizes:
    print(f"Размер чанка (симв.): мин {min(sizes)}, средн. {sum(sizes)//len(sizes)}, макс {max(sizes)}")

In [ ]:
# Смотрим 3 случайных чанка целиком
import random
for c in random.sample(chunks, min(3, len(chunks))):
    print("=" * 70)
    print(c["link"], "| сообщения", c["first_msg_id"], "-", c["last_msg_id"])
    print(c["text"])

## 5. Индексация (эмбеддинги → Chroma)

⚠️ Этот шаг тратит токены OpenAI (эмбеддинги дёшевые, но не бесплатные). Запускай, когда чанки выглядят хорошо.

In [ ]:
# Обычно индексацию удобнее запускать скриптом (читает data/chunks/*.jsonl):
#   uv run python -m src.preprocess   # сохранить чанки на диск
#   uv run python -m src.index        # построить индекс
#
# Или прямо здесь:
from src.index import main as build_index
from src.preprocess import main as build_chunks

build_chunks()   # data/chunks/*.jsonl
build_index()    # эмбеддинги -> chroma_db/

In [ ]:
from src.store import get_collection
col = get_collection()
print("Записей в коллекции:", col.count())

## 6. Поиск (retrieve)

Смотрим, какие фрагменты находятся по вопросу и насколько они релевантны (score ближе к 1 = лучше).

In [ ]:
from src.retrieve import search

query = "как открыть ип в грузии"  # <- меняй вопрос

for i, h in enumerate(search(query, k=5), 1):
    print(f"--- #{i}  score={h['score']:.3f}  {h['meta']['link']}")
    print(h["text"][:300])
    print()

## 7. Ответ RAG

Финальный ответ от GPT по найденным фрагментам + список источников.

In [ ]:
from src.rag import answer

res = answer("какие документы нужны для открытия ип")  # <- меняй вопрос

print(res["answer"])
print("\n——— Источники ———")
for s in res["sources"]:
    print(s["title"], "|", s["link"])

## 8. Дебаг: какой промпт реально уходит в GPT

Полезно, чтобы понять, почему модель ответила именно так, и подкрутить системный промпт в `src/rag.py`.

In [ ]:
from src.rag import _build_context, SYSTEM_PROMPT
from src.retrieve import search

q = "как получить внж"
hits = search(q, k=4)

print("### SYSTEM PROMPT ###\n")
print(SYSTEM_PROMPT)
print("\n### КОНТЕКСТ (фрагменты) ###\n")
print(_build_context(hits))